# 01 — Data Exploration

This notebook loads and explores the **Pantheon+SH0ES** Type Ia supernova dataset.

We examine:
- The redshift distribution of the 1701 supernovae
- The Hubble diagram (distance modulus vs redshift)
- The covariance matrix structure
- The diagonal error distribution

> **Data source:** Scolnic et al. 2022, Brout et al. 2022, Riess et al. 2022  
> See `data/README.md` for download instructions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

plt.rcParams.update({'font.size': 12})
print('Libraries loaded.')

## 1.1 Load the Dataset

In [ ]:
data = pd.read_csv('../Pantheon+SH0ES_data.dat', sep=r'\s+')
data = data.replace([np.inf, -np.inf], np.nan)
data = data.dropna(subset=['zHD', 'MU_SH0ES', 'MU_SH0ES_ERR_DIAG'])

mask = data['zHD'].values > 1e-4
data = data[mask].reset_index(drop=True)

z      = data['zHD'].values
mu_obs = data['MU_SH0ES'].values
mu_err = data['MU_SH0ES_ERR_DIAG'].values

print(f'Total supernovae: {len(z)}')
print(f'Redshift range  : {z.min():.4f} -- {z.max():.4f}')
print(f'mu range        : {mu_obs.min():.2f} -- {mu_obs.max():.2f} mag')
data.head()

## 1.2 Redshift Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(z, bins=60, color='steelblue', edgecolor='white',
        linewidth=0.4, alpha=0.85)
ax.set_xlabel('Redshift $z$', fontsize=13)
ax.set_ylabel('Number of SNe Ia', fontsize=13)
ax.set_title(f'Redshift Distribution — Pantheon+ ({len(z)} SNe Ia)',
             fontsize=13)
ax.axvline(np.median(z), color='crimson', linestyle='--',
           linewidth=1.8, label=f'Median $z$ = {np.median(z):.3f}')
ax.legend(fontsize=11)
ax.grid(axis='y', linestyle=':', alpha=0.4)
plt.tight_layout()
plt.savefig('../figures/fig_redshift_dist.pdf', bbox_inches='tight')
plt.show()

## 1.3 The Hubble Diagram

The Hubble diagram plots distance modulus $\mu$ vs redshift $z$.
Each point is a Type Ia supernova. The scatter around the theoretical
curve reflects both measurement uncertainty and intrinsic scatter.

We overplot the flat $\Lambda$CDM prediction for two reference cosmologies.

In [ ]:
from scipy.integrate import quad

c = 299792.458  # km/s

def mu_theory(z_arr, H0, Om):
    OL = 1.0 - Om
    chi = np.array([quad(lambda zp: 1/np.sqrt(Om*(1+zp)**3 + OL),
                        0, zi)[0] for zi in z_arr])
    dl = (c / H0) * (1 + z_arr) * chi
    return 5 * np.log10(dl) + 25

z_th = np.linspace(0.001, z.max(), 300)

fig, axes = plt.subplots(2, 1, figsize=(10, 8),
                         gridspec_kw={'height_ratios': [3, 1]},
                         sharex=True)

# Hubble diagram
axes[0].errorbar(z, mu_obs, yerr=mu_err, fmt='.', color='steelblue',
                 alpha=0.3, markersize=3, elinewidth=0.4,
                 label='Pantheon+ SNe Ia')
mu_shoes = mu_theory(z_th, 73.04, 0.334)
mu_planck = mu_theory(z_th, 67.4, 0.315)
axes[0].plot(z_th, mu_shoes,  color='crimson',  lw=2,
             label=r'SH0ES 2022: $H_0$=73.04, $\Omega_m$=0.334')
axes[0].plot(z_th, mu_planck, color='teal', lw=2, linestyle='--',
             label=r'Planck 2018: $H_0$=67.4, $\Omega_m$=0.315')
axes[0].set_ylabel(r'Distance Modulus $\mu$ (mag)', fontsize=13)
axes[0].set_title('Hubble Diagram — Pantheon+SH0ES', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(linestyle=':', alpha=0.3)

# Residuals
mu_ref = mu_theory(z, 73.04, 0.334)
axes[1].errorbar(z, mu_obs - mu_ref, yerr=mu_err, fmt='.',
                 color='steelblue', alpha=0.3, markersize=3,
                 elinewidth=0.4)
axes[1].axhline(0, color='crimson', lw=1.5)
axes[1].set_xlabel('Redshift $z$', fontsize=13)
axes[1].set_ylabel(r'$\Delta\mu$ (mag)', fontsize=13)
axes[1].grid(linestyle=':', alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/fig_hubble_diagram.pdf', bbox_inches='tight')
plt.show()

## 1.4 Covariance Matrix Structure

The Pantheon+ systematic covariance matrix encodes correlated
uncertainties across all 1701 supernovae. We visualise its structure
and compare the diagonal to the reported statistical errors.

In [ ]:
print('Loading covariance matrix...')
with open('../Pantheon+SH0ES_STAT+SYS (4).cov', 'r') as f:
    n_cov  = int(f.readline().strip())
    C_full = np.array(f.read().split(), dtype=float).reshape(n_cov, n_cov)

print(f'Covariance matrix size: {n_cov} x {n_cov}')
print(f'sqrt(C_diag) range: [{np.sqrt(np.diag(C_full)).min():.4f}, '
      f'{np.sqrt(np.diag(C_full)).max():.4f}]')
print(f'MU_ERR_DIAG  range: [{mu_err.min():.4f}, {mu_err.max():.4f}]')
print()
print('Total covariance C = C_sys + diag(sigma_stat^2)')
C_total = C_full + np.diag(mu_err**2)
print(f'sqrt(C_total diag) range: [{np.sqrt(np.diag(C_total)).min():.4f}, '
      f'{np.sqrt(np.diag(C_total)).max():.4f}]')

In [ ]:
# Visualise a submatrix of the covariance
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(C_full[:100, :100], cmap='RdBu_r',
                     norm=mcolors.SymLogNorm(linthresh=1e-4))
plt.colorbar(im0, ax=axes[0])
axes[0].set_title('Systematic Covariance $C_{sys}$ (first 100x100)',
                  fontsize=11)

axes[1].hist(np.sqrt(np.diag(C_total)), bins=50,
             color='steelblue', edgecolor='white', linewidth=0.4)
axes[1].set_xlabel(r'$\sqrt{C_{total,ii}}$ (mag)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Distribution of Total Diagonal Uncertainties',
                  fontsize=11)
axes[1].grid(axis='y', linestyle=':', alpha=0.4)

plt.tight_layout()
plt.savefig('../figures/fig_covariance.pdf', bbox_inches='tight')
plt.show()

## Summary

| Property | Value |
|----------|-------|
| Total SNe Ia | 1701 |
| Redshift range | 0.00122 – 2.26 |
| Covariance matrix | 1701 × 1701 |
| Total covariance | $C = C_{sys} + \mathrm{diag}(\sigma_{stat}^2)$ |

The dataset is clean and ready for analysis. Proceed to `02_frequentist_mle.ipynb`.